In [ ]:
#!/usr/bin/env python
# coding: utf-8
"""
================================================================================
 GC_TAM_Figure3.py  --  gene-regulatory networks and trajectories of GC myeloid cells

 Visual template : Motevasseli et al., Acta Neuropathol Commun (2024) 12:133, Figure 3
                   A regulon heatmap | B,C RSS plots | D regulon-activity UMAPs |
                   E Palantir pseudotime | F CellRank streamlines |
                   G,H lineage-driver heatmaps | I gene trends along pseudotime
 Extensions that tie Figure 3 to the Figure-1 axis (SAP, stromal activation programme):
                   J regulon activity vs SAP WITHIN each state (tumour level)
                   K fate bias of monocytes/transitional cells vs SAP (tumour level)

 INPUT (all produced by GC_TAM_Stage1_Figure1.py; nothing is recomputed upstream)
   h5ad/S1e_myeloid_final_v3_m3_amb20_F4.h5ad   13,142 tumour myeloid cells, raw counts in
                                                 layers["counts"], log-normalised X, X_scVI, UMAP
   tables/S2_MCP_scores_v3_amb20_F4.csv          SAP score per tumour
   tables/Fig1E_composition_vs_axis_m3_amb20_F4.csv   SAP-associated states (data-driven choice
                                                 of the states shown in B/C and G/H)
   resources/  allTFs_hg38.txt, 2 cisTarget feather DBs, motif annotation table

 METHODS (Q1-level safeguards)
   * pySCENIC on raw counts. GRNBoost2: regulators = motif-annotated TFs only (a TF without a
     motif cannot pass cisTarget -> lossless), targets = 3,000 HVGs U those TFs; each of the
     N_GRN_RUNS runs uses a different seed AND an independent state-stratified cell subsample
     (<= 1,000 cells/state, rare states whole) -> consensus over seeds and subsamples (bagging).
     A regulon is kept if its TF is recovered in >= CONSENSUS_MIN_RUNS runs, a target gene if
     recovered for that TF in >= CONSENSUS_MIN_RUNS runs; regulons < 10 genes discarded.
     cisTarget uses ALL cells (TF-target correlation sign).
   * AUCell on ALL cells, ranking ALL detected genes (top 5%, pySCENIC default) on the consensus
     regulons;
     RSS (Jensen-Shannon) per state.
   * SAP association of regulons is tested WITHIN states (tumour pseudobulk of AUC), so it is not
     driven by changes in state composition; SAP is rank-inverse-normal transformed (robust to the
     single extreme tumour, Pat01-B, SAP z = -6.3); dataset is a covariate; global BH-FDR.
   * Trajectory restricted to the monocyte-derived lineage (classical monocytes -> transitional ->
     inflammatory / SPP1 / C1QC TAMs). Proliferating, IFN, DC and neutrophil states are excluded
     (cell-cycle and interferon programmes are activation states orthogonal to differentiation).
     Start cell = data-driven (extreme monocyte identity score); robustness to the start cell is
     reported. CellRank 2: 0.8 PseudotimeKernel + 0.2 ConnectivityKernel, GPCCA fate probabilities.

 RUN:  python GC_TAM_Figure3.py   (checkpointed; GRNBoost2 is the slow step)
================================================================================
"""

# ==============================================================================
# 0. IMPORTS + COMPATIBILITY PATCHES (pySCENIC 0.12.1 / numpy 2 / pandas 2 / CellRank 2)
# ==============================================================================
import os
import re
import sys
import json
import time
import subprocess
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy import stats
from scipy.ndimage import gaussian_filter1d

for _n, _v in {"object": object, "bool": bool, "int": int, "float": float,
               "complex": complex, "str": str}.items():
    if _n not in np.__dict__:
        setattr(np, _n, _v)
if not hasattr(pd.Series, "iteritems"):
    pd.Series.iteritems = pd.Series.items
for _cls in [getattr(sp, n, None) for n in ["spmatrix", "csr_matrix", "csc_matrix", "coo_matrix",
                                            "lil_matrix", "dok_matrix", "bsr_matrix", "dia_matrix"]]:
    if _cls is not None and not hasattr(_cls, "A"):
        _cls.A = property(lambda self: self.toarray())

import scanpy as sc
import anndata as ad
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import gridspec
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from adjustText import adjust_text

# jax is OPTIONAL for CellRank (only its velocity-kernel similarity uses it) and is not used here.
# In this environment jax -> ml_dtypes 0.6 crashes at import with the installed NumPy
# ("ufunc 'isnan' not supported"), and CellRank only catches ImportError. Blocking jax makes
# `import jax` raise ImportError, so CellRank falls back to its NumPy implementation.
# Nothing in this script (PseudotimeKernel, ConnectivityKernel, GPCCA, Palantir) needs jax.
for _m in ["jax", "jax.numpy", "jaxlib", "ml_dtypes"]:
    sys.modules[_m] = None

import palantir
import cellrank as cr

# Unblock again once CellRank/Palantir are imported: CellRank has already fixed its NumPy fallback.
# If the None placeholders stayed in sys.modules, scipy's array-API layer (used by pearsonr inside
# Palantir) would see "jax" in sys.modules and try `import jax`, which fails. With the entries
# removed, scipy correctly concludes that jax is not in use.
for _m in ["jax", "jax.numpy", "jaxlib", "ml_dtypes"]:
    if sys.modules.get(_m, 0) is None:
        del sys.modules[_m]


def _solver_compat(fn):
    """CellRank calls scipy iterative solvers with tol=/atol='legacy' (removed in new scipy)."""
    def _wrapped(*args, **kwargs):
        kw = dict(kwargs)
        for _ in range(4):
            try:
                return fn(*args, **kw)
            except (TypeError, ValueError) as e:
                msg, changed = str(e), False
                if "tol" in kw and "'tol'" in msg:
                    kw["rtol"] = kw.pop("tol"); changed = True
                if "atol" in kw and "atol" in msg:
                    kw.pop("atol", None); changed = True
                if not changed:
                    raise
        return fn(*args, **kw)
    return _wrapped


try:
    import cellrank._utils._linear_solver as _crls
    if hasattr(_crls, "_AVAIL_ITER_SOLVERS"):
        _crls._AVAIL_ITER_SOLVERS = {k: _solver_compat(v) for k, v in _crls._AVAIL_ITER_SOLVERS.items()}
except Exception as _e:
    print("[warning] CellRank solver patch not applied:", _e)

from pyscenic.utils import load_motifs
from pyscenic.transform import df2regulons
from pyscenic.rss import regulon_specificity_scores

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# ==============================================================================
# 1. CONFIGURATION
# ==============================================================================
BASE = Path(os.environ.get("GC_BASE", "/home/arshia/payanname"))
IN = BASE / "GC_Fig1_outputs"
RES = IN / "resources"
MY_H5AD = IN / "h5ad" / "S1e_myeloid_final_v3_m3_amb20_F4.h5ad"
SAP_CSV = IN / "tables" / "S2_MCP_scores_v3_amb20_F4.csv"
COMP_CSV = IN / "tables" / "Fig1E_composition_vs_axis_m3_amb20_F4.csv"

OUT = BASE / "GC_Fig3_outputs"
SCN, TAB, H5 = OUT / "scenic", OUT / "tables", OUT / "h5ad"
FIG_MAIN, FIG_SUPP = OUT / "figures" / "main", OUT / "figures" / "supp"
for _d in [OUT, SCN, TAB, H5, FIG_MAIN, FIG_SUPP]:
    _d.mkdir(parents=True, exist_ok=True)
LOGFILE = OUT / "run_log.txt"

TF_LIST = RES / "allTFs_hg38.txt"
RANKING_DBS = [RES / "hg38_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather",
               RES / "hg38_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather"]
MOTIF_TBL = RES / "motifs-v10nr_clust-nr.hgnc-m0.001-o0.0.tbl"

SEED = 0
N_WORKERS = min(2, max(1, (os.cpu_count() or 4) - 1))   # 2 workers: enough RAM per worker
AXIS_LABEL = "SAP"

# --- pySCENIC ---
MIN_CELL_FRAC = 0.01            # genes detected in >= 1% of myeloid cells
# --- speed-ups that do not change what can be learned (Q1-defensible) ---
# (1) Regulators: only TFs that HAVE a motif in the cisTarget annotation. A TF without a motif can
#     never survive cisTarget pruning, so removing it from GRNBoost2 is lossless.
# (2) Targets: 3,000 HVGs U motif-annotated TFs (regulons are built from co-varying genes; low-variance
#     genes contribute no co-expression signal to GRNBoost2).
# (3) Cells: each GRNBoost2 run uses an independent state-stratified subsample (<= 1,000 cells per
#     state; rare states kept whole), so the consensus across runs is also a consensus across cell
#     subsamples (bagging). cisTarget (activating/repressing sign) and AUCell use ALL cells, and AUCell
#     ranks ALL detected genes (SCENIC protocol), so every cell receives regulon scores.
N_HVG_GRN = 3000
GRN_CELLS_PER_STATE = 1000
N_GRN_RUNS, CONSENSUS_MIN_RUNS = 3, 2
SCN_TAG = "v2"                  # checkpoint names for the accelerated configuration
MIN_REGULON_SIZE = 10
AUC_TOP_FRAC = 0.05             # pySCENIC AUCell default
EXCLUDE_GENE_RE = re.compile(r"^(MT-|RP[SL]\d|HB[ABDEGMQZ]\d?$|MTRNR|IG[HKL]V|TR[ABDG]V)")

# --- regulon / fate vs SAP (tumour level) ---
MIN_CELLS_STATE_TUMOUR = 10
N_TOP_RSS_PER_STATE = 5

# --- trajectory ---
TRAJ_STATES = ["Mono_classical", "TAM_Transitional", "TAM_Inflammatory", "TAM_SPP1", "TAM_C1QC"]
ORIGIN_STATE = "Mono_classical"
TERMINAL_STATES = ["TAM_Inflammatory", "TAM_SPP1", "TAM_C1QC"]
N_WAYPOINTS = 1200
TERMINAL_TOP_FRAC = 0.10        # CellRank terminal cells: top 10% pseudotime within the state
PK_WEIGHT, CK_WEIGHT = 0.8, 0.2
N_DRIVERS = 20
TREND_GENES_FIXED = ["CCR2", "SLAMF9", "IL1B", "C1QC"]   # CCR2: receptor of the SAP ligand CCL7;
                                                          # SLAMF9/IL1B: SAP-high DEGs (Fig1G)
FORCE = set()                   # e.g. {"grn", "ctx", "consensus", "aucell", "traj"}

MONO_GENES = ["FCN1", "VCAN", "S100A8", "S100A9", "S100A12", "CD300E"]
MACRO_GENES = ["C1QA", "C1QB", "C1QC", "APOE"]
STATE_PANELS = {
    "TAM_SPP1": ["SPP1", "GPNMB", "FBP1", "MMP9", "MMP12", "CHI3L1"],
    "TAM_C1QC": ["C1QA", "C1QB", "C1QC", "CCL18", "MS4A6A"],
    "TAM_Inflammatory": ["IL1B", "CXCL8", "CCL3", "CCL4", "EREG", "NLRP3", "TNF"],
}

# ==============================================================================
# 2. STYLE (identical to Figure 1; template Fig. 3 colour schemes)
# ==============================================================================
def _pick_font():
    from matplotlib import font_manager
    avail = {f.name for f in font_manager.fontManager.ttflist}
    for f in ["Arial", "Helvetica", "Liberation Sans", "Nimbus Sans", "DejaVu Sans"]:
        if f in avail:
            return f
    return "DejaVu Sans"


mpl.rcParams.update({
    "font.family": "sans-serif", "font.sans-serif": [_pick_font()],
    "font.size": 7, "axes.titlesize": 8, "axes.titleweight": "bold",
    "axes.labelsize": 7, "xtick.labelsize": 6, "ytick.labelsize": 6,
    "legend.fontsize": 6, "legend.title_fontsize": 6.5,
    "axes.linewidth": 0.6, "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "xtick.major.size": 2.5, "ytick.major.size": 2.5, "lines.linewidth": 0.8,
    "pdf.fonttype": 42, "ps.fonttype": 42, "svg.fonttype": "none",
    "savefig.dpi": 600, "figure.dpi": 150, "savefig.bbox": "tight", "savefig.pad_inches": 0.02,
})
sc.settings.verbosity = 1

MYELOID_PALETTE = {
    "TAM_C1QC": "#6A3D9A", "TAM_FOLR2": "#33A02C", "TAM_SPP1": "#E31A1C",
    "TAM_Inflammatory": "#FF9E4A", "TAM_IFN": "#F2D400", "TAM_Prolif": "#1A1A1A",
    "TAM_Transitional": "#A6761D", "Mono_classical": "#8C2D19", "Mono_nonclassical": "#C98B63",
    "cDC1": "#1F5FA8", "cDC2": "#4F8FD6", "mregDC": "#9ECAE9", "pDC": "#8C8C8C", "Neutrophil": "#BCBD22",
}
MYELOID_ORDER = ["Mono_classical", "TAM_Transitional", "TAM_Inflammatory", "TAM_SPP1", "TAM_C1QC",
                 "TAM_IFN", "TAM_Prolif", "cDC1", "cDC2", "mregDC", "pDC", "Neutrophil"]
DATASET_PALETTE = {"GSE183904": "#4A7EB5", "GSE150290": "#E3893F"}
AUC_CMAP = LinearSegmentedColormap.from_list(              # template Fig. 3D: white-yellow-red-black
    "auc", ["#FFFFFF", "#FFF3A6", "#FFC300", "#F0590E", "#A10F0F", "#000000"])
RSS_BLUE, RSS_RED, RSS_GREEN = "#4C6FB5", "#D62728", "#1E7B1E"


def log(msg):
    line = f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] {msg}"
    print(line, flush=True)
    with open(LOGFILE, "a") as fh:
        fh.write(line + "\n")


def savefig(fig, name, folder=FIG_MAIN):
    for ext in ["pdf", "svg", "png"]:
        fig.savefig(folder / f"{name}.{ext}")
    plt.close(fig)
    log(f"  saved figure {folder.name}/{name}")


def panel_label(ax, letter, x=-0.02, y=1.02):
    ax.text(x, y, letter, fontsize=10, fontweight="bold", va="bottom", ha="right", transform=ax.transAxes)


def rasterize_points(ax):
    for coll in ax.collections:
        coll.set_rasterized(True)


def umap_axis_arrows(ax, label="UMAP"):
    kw = dict(xycoords="axes fraction", textcoords="axes fraction",
              arrowprops=dict(arrowstyle="-|>", lw=0.6, color="black", mutation_scale=6))
    ax.annotate("", xy=(0.16, 0.02), xytext=(0.02, 0.02), **kw)
    ax.annotate("", xy=(0.02, 0.16), xytext=(0.02, 0.02), **kw)
    ax.text(0.02, -0.02, f"{label} 1", transform=ax.transAxes, fontsize=5.5, va="top", ha="left")
    ax.text(-0.01, 0.02, f"{label} 2", transform=ax.transAxes, fontsize=5.5, va="bottom", ha="right",
            rotation=90)


def set_palette(adata, key):
    adata.obs[key] = pd.Categorical(adata.obs[key].astype(str),
                                    categories=[s for s in MYELOID_ORDER if s in set(adata.obs[key].astype(str))])
    adata.uns[f"{key}_colors"] = [MYELOID_PALETTE.get(c, "#BBBBBB") for c in adata.obs[key].cat.categories]


def _get_read_elem():
    for mod, name in [("anndata.io", "read_elem"), ("anndata.experimental", "read_elem"),
                      ("anndata._io.specs", "read_elem")]:
        try:
            return getattr(__import__(mod, fromlist=[name]), name)
        except Exception:
            continue
    raise ImportError("anndata read_elem not found")


def read_h5ad_compat(path):
    """Read an h5ad written by a NEWER anndata (0.12, used for Figure 1) in this older environment.
    Newer anndata encodes None values as IOSpec 'null' (e.g. uns['log1p']['base']), which older
    readers cannot parse. Every element is read separately; elements the old reader cannot parse
    are skipped with a log entry -- only unstructured metadata in `uns` is affected (X, layers,
    obs, var, obsm, obsp are standard encodings)."""
    try:
        return sc.read_h5ad(path)
    except Exception as e:
        log(f"  standard read failed ({type(e).__name__}); using element-wise compatible reader")
    import h5py
    read_elem = _get_read_elem()

    def read_df(g, what):
        try:
            return read_elem(g)
        except Exception:
            index = pd.Index(np.asarray(read_elem(g[g.attrs.get("_index", "_index")])).astype(str))
            data = {}
            for c in list(g.attrs.get("column-order", [])):
                try:
                    data[c] = read_elem(g[c])
                except Exception as ex:
                    log(f"    skipped {what} column '{c}' ({type(ex).__name__})")
            return pd.DataFrame(data, index=index)

    def read_group(f, key):
        out = {}
        if key not in f:
            return out
        for k in f[key].keys():
            try:
                out[k] = read_elem(f[key][k])
            except Exception as ex:
                log(f"    skipped {key}['{k}'] ({type(ex).__name__})")
        return out

    with h5py.File(path, "r") as f:
        X = read_elem(f["X"])
        obs, var = read_df(f["obs"], "obs"), read_df(f["var"], "var")
        a = ad.AnnData(X=X, obs=obs, var=var, layers=read_group(f, "layers"),
                       obsm=read_group(f, "obsm"), obsp=read_group(f, "obsp"))
        uns = {}
        if "uns" in f:
            for k in f["uns"].keys():
                try:
                    uns[k] = read_elem(f["uns"][k])
                except Exception:
                    if k == "log1p":
                        uns[k] = {"base": None}          # the only null-encoded field written by Fig1
                    else:
                        log(f"    skipped uns['{k}']")
        a.uns = uns
    return a


def safe_write(adata, path):
    """Write h5ad, converting objects that anndata cannot serialise (Palantir/CellRank leave
    pandas Index / DataFrame / custom objects in uns and obsm)."""
    for k in list(adata.obsm.keys()):
        if isinstance(adata.obsm[k], pd.DataFrame):
            adata.uns[f"{k}_columns"] = np.asarray(adata.obsm[k].columns.astype(str))
            adata.obsm[k] = adata.obsm[k].to_numpy()
    for k in list(adata.uns.keys()):
        v = adata.uns[k]
        if isinstance(v, (pd.Index, pd.Series)):
            adata.uns[k] = np.asarray(v.astype(str) if v.dtype == object else v)
        elif not isinstance(v, (np.ndarray, str, int, float, bool, list, dict, pd.DataFrame)):
            del adata.uns[k]
    try:
        adata.write_h5ad(path, compression="gzip")
    except Exception as e:
        log(f"  [warning] full write failed ({e}); retrying without uns")
        adata.uns = {k: v for k, v in adata.uns.items() if k.endswith("_colors")}
        adata.write_h5ad(path, compression="gzip")


def fdr_stars(q):
    return "***" if q < 0.001 else "**" if q < 0.01 else "*" if q < 0.05 else "ns"


def tf_of(regulon_name):
    return regulon_name.split("(")[0]


def rank_inverse_normal(x):
    r = stats.rankdata(x)
    return pd.Series(stats.norm.ppf((r - 0.375) / (len(x) + 0.25)), index=x.index)


def aucell(X, gene_names, gene_sets, top_frac=AUC_TOP_FRAC, chunk=2000, seed=SEED):
    """AUCell (Aibar et al., 2017) -- identical to the Figure-1 implementation."""
    X = X.tocsr() if sp.issparse(X) else sp.csr_matrix(X)
    n_genes = X.shape[1]
    max_rank = int(np.ceil(top_frac * n_genes))
    perm = np.random.default_rng(seed).permutation(n_genes)
    gidx = pd.Index(gene_names)
    members, n_eff = {}, {}
    for name, genes in gene_sets.items():
        m = np.zeros(n_genes, dtype=np.float32)
        pos = gidx.get_indexer([g for g in genes if g in gidx])
        pos = pos[pos >= 0]
        m[pos] = 1.0
        members[name], n_eff[name] = m[perm], max(len(pos), 1)
    weights = (max_rank - np.arange(max_rank)).astype(np.float32)
    out = {k: np.zeros(X.shape[0], dtype=np.float32) for k in gene_sets}
    for s in range(0, X.shape[0], chunk):
        D = X[s:s + chunk].toarray()[:, perm]
        top_idx = np.argpartition(-D, max_rank - 1, axis=1)[:, :max_rank]
        order = np.take_along_axis(D, top_idx, axis=1)
        top_idx = np.take_along_axis(top_idx, np.argsort(-order, axis=1, kind="stable"), axis=1)
        for k, m in members.items():
            out[k][s:s + chunk] = m[top_idx] @ weights / (min(n_eff[k], max_rank) * max_rank)
    return pd.DataFrame(out)


# ==============================================================================
# 3. INPUTS
# ==============================================================================
def load_inputs():
    my = read_h5ad_compat(MY_H5AD)
    for need in ["myeloid_state", "sample_id", "dataset"]:
        if need not in my.obs:
            raise RuntimeError(f"obs['{need}'] could not be read from {MY_H5AD}")
    if "counts" not in my.layers or "X_scVI" not in my.obsm or "X_umap" not in my.obsm:
        raise RuntimeError("layers['counts'] / obsm['X_scVI'] / obsm['X_umap'] missing after read")
    my.obs["myeloid_state"] = my.obs["myeloid_state"].astype(str)
    sap = pd.read_csv(SAP_CSV, index_col=0)
    sap["SAP_rin"] = rank_inverse_normal(sap["MCP"])
    comp = pd.read_csv(COMP_CSV)
    up = comp[(comp.fdr < 0.05) & (comp.beta_clr_per_SD > 0)].sort_values("beta_clr_per_SD", ascending=False)
    dn = comp[(comp.fdr < 0.05) & (comp.beta_clr_per_SD < 0)].sort_values("beta_clr_per_SD")
    sap_high, sap_low = up.state.tolist(), dn.state.tolist()
    log(f"Inputs: {my.n_obs:,} myeloid cells; SAP-high states {sap_high}; SAP-low states {sap_low}")
    return my, sap, sap_high, sap_low


# ==============================================================================
# 4. pySCENIC (consensus of N_GRN_RUNS GRNBoost2 runs)
# ==============================================================================
def motif_tfs():
    cache = SCN / "motif_annotated_tfs.txt"
    if cache.exists():
        return set(pd.read_csv(cache, header=None)[0].astype(str))
    ann = pd.read_csv(MOTIF_TBL, sep="\t", dtype=str)
    col = [c for c in ann.columns if c.lstrip("#").lower() == "gene_name"][0]
    tfs = set(ann[col].dropna().astype(str))
    pd.Series(sorted(tfs)).to_csv(cache, index=False, header=False)
    return tfs


def scenic_matrix(my):
    """Raw counts. Detected genes (>=1% of cells) -> AUCell universe; GRN universe = 3,000 HVGs U
    motif-annotated TFs."""
    out = SCN / f"grn_input_genes_{SCN_TAG}.txt"
    tfs_all = set(pd.read_csv(TF_LIST, header=None)[0].astype(str)) & motif_tfs()
    X = my.layers["counts"].tocsc() if sp.issparse(my.layers["counts"]) else sp.csc_matrix(my.layers["counts"])
    det = np.asarray((X > 0).sum(axis=0)).ravel() / my.n_obs
    keep = (det >= MIN_CELL_FRAC) & ~my.var_names.str.match(EXCLUDE_GENE_RE)
    genes = my.var_names[keep]
    if out.exists() and "grn" not in FORCE:
        target = pd.read_csv(out, header=None)[0].tolist()
    else:
        tmp = ad.AnnData(X=X[:, keep].tocsr(), obs=my.obs[["dataset"]].copy(), var=pd.DataFrame(index=genes))
        tmp.layers["counts"] = tmp.X.copy()
        sc.pp.highly_variable_genes(tmp, n_top_genes=N_HVG_GRN, flavor="seurat_v3", layer="counts",
                                    batch_key="dataset")
        hvg = set(tmp.var_names[tmp.var.highly_variable])
        target = sorted(hvg | (tfs_all & set(genes)))
        pd.Series(target).to_csv(out, index=False, header=False)
    tfs = sorted(tfs_all & set(target))
    M = X[:, my.var_names.get_indexer(target)].tocsr()
    M_all = X[:, np.where(keep)[0]].tocsr()
    log(f"SCENIC: GRN universe {len(target):,} genes ({len(tfs)} motif-annotated TFs); "
        f"AUCell universe {len(genes):,} detected genes; {my.n_obs:,} cells")
    return M, target, tfs, M_all, list(genes)


def grn_subsample(states, seed):
    """State-stratified subsample: <= GRN_CELLS_PER_STATE per state, rare states kept whole."""
    rng = np.random.default_rng(1000 + seed)
    idx = []
    for s_ in pd.unique(states):
        ix = np.where(states == s_)[0]
        idx.extend(ix if len(ix) <= GRN_CELLS_PER_STATE else rng.choice(ix, GRN_CELLS_PER_STATE, replace=False))
    return np.sort(np.asarray(idx))


def write_loom(M, genes, cells, path):
    import loompy as lp
    if path.exists() and path.stat().st_size > 1000:
        return
    lp.create(str(path), M.T.tocsr().astype(np.float32).toarray(),
              {"Gene": np.asarray(genes)}, {"CellID": np.asarray(cells)})


def run_grnboost2(M, genes, tfs, seed, out_tsv):
    """GRNBoost2 via arboreto.core.create_graph(include_meta=True): avoids the known
    arboreto / dask-expr crash (from_delayed([]) in grnboost2())."""
    if out_tsv.exists() and out_tsv.stat().st_size > 1000 and "grn" not in FORCE:
        return
    from distributed import Client, LocalCluster
    from arboreto.core import create_graph, SGBM_KWARGS
    df = pd.DataFrame(M.toarray().astype(np.float32), columns=genes)
    cluster = LocalCluster(n_workers=N_WORKERS, threads_per_worker=1, processes=False,
                           dashboard_address=None, silence_logs=True)
    client = Client(cluster)
    try:
        log(f"  GRNBoost2 seed={seed}: {df.shape[0]:,} cells x {df.shape[1]:,} genes, {len(tfs)} TFs")
        graph = create_graph(df.values, list(df.columns), tfs, client=client, regressor_type="GBM",
                             regressor_kwargs=SGBM_KWARGS, early_stop_window_length=25, limit=None,
                             include_meta=True, seed=seed)
        res = client.compute(graph, sync=True)
        if isinstance(res, (list, tuple)):
            res = next(r for r in res if hasattr(r, "columns") and "importance" in r.columns)
        res.sort_values("importance", ascending=False).to_csv(out_tsv, sep="\t", index=False)
    finally:
        client.close()
        cluster.close()


def run_pyscenic_cli(argv):
    script = "\n".join([
        "import numpy as np",
        "for _n, _v in {'object': object, 'bool': bool, 'int': int, 'float': float, 'complex': complex, 'str': str}.items():",
        "    if _n not in np.__dict__: setattr(np, _n, _v)",
        "import pandas as pd",
        "if not hasattr(pd.Series, 'iteritems'): pd.Series.iteritems = pd.Series.items",
        "import sys",
        "from pyscenic.cli.pyscenic import main",
        f"sys.argv = {[str(a) for a in argv]!r}",
        "main()",
    ])
    env = os.environ.copy()
    env.pop("DASK_DATAFRAME__QUERY_PLANNING", None)
    subprocess.run([sys.executable, "-c", script], check=True, env=env)


def run_ctx(adj_tsv, loom, out_csv):
    if out_csv.exists() and out_csv.stat().st_size > 1000 and "ctx" not in FORCE:
        return
    log(f"  cisTarget pruning: {adj_tsv.name}")
    run_pyscenic_cli(["pyscenic", "ctx", adj_tsv, *RANKING_DBS, "--annotations_fname", MOTIF_TBL,
                      "--expression_mtx_fname", loom, "--mode", "custom_multiprocessing",
                      "--output", out_csv, "--num_workers", N_WORKERS])


def consensus_regulons(run_csvs):
    out = TAB / f"Fig3_consensus_regulons_{SCN_TAG}.csv"
    if out.exists() and "consensus" not in FORCE:
        t = pd.read_csv(out)
        return {r.regulon: r.genes.split(";") for r in t.itertuples()}, t
    per_run = []
    for f in run_csvs:
        regs = df2regulons(load_motifs(str(f)))
        d = {}
        for r in regs:
            d.setdefault(tf_of(r.name), set()).update(r.gene2weight.keys())
        per_run.append(d)
    tfs = sorted(set().union(*[set(d) for d in per_run]))
    rows, gs = [], {}
    for tf in tfs:
        found = [d[tf] for d in per_run if tf in d]
        if len(found) < CONSENSUS_MIN_RUNS:
            continue
        cnt = pd.Series([g for s in found for g in s]).value_counts()
        genes = sorted(cnt.index[cnt >= CONSENSUS_MIN_RUNS])
        if tf not in genes:
            genes.append(tf)                             # TF is part of its own regulon (pySCENIC)
        if len(genes) < MIN_REGULON_SIZE:
            continue
        name = f"{tf}(+)"
        gs[name] = genes
        rows.append(dict(regulon=name, tf=tf, n_runs=len(found), n_genes=len(genes),
                         jaccard_mean=np.mean([len(a & b) / len(a | b) for i, a in enumerate(found)
                                               for b in found[i + 1:]]) if len(found) > 1 else np.nan,
                         genes=";".join(genes)))
    t = pd.DataFrame(rows).sort_values("n_genes", ascending=False)
    t.to_csv(out, index=False)
    log(f"  consensus regulons: {len(t)} (TF in >= {CONSENSUS_MIN_RUNS}/{len(run_csvs)} runs, "
        f">= {MIN_REGULON_SIZE} genes)")
    return gs, t


def run_scenic(my):
    M, genes, tfs, M_all, genes_all = scenic_matrix(my)
    loom = SCN / f"myeloid_counts_{SCN_TAG}.loom"
    write_loom(M, genes, my.obs_names, loom)                 # all cells (sign of each TF-target link)
    states = my.obs["myeloid_state"].astype(str).values
    run_csvs, sub_rows = [], []
    for i in range(N_GRN_RUNS):
        adj = SCN / f"adjacencies_{SCN_TAG}_seed{i}.tsv"
        reg = SCN / f"regulons_{SCN_TAG}_seed{i}.csv"
        sub = grn_subsample(states, i)
        sub_rows.append(pd.Series(states[sub]).value_counts().rename(f"run{i}"))
        run_grnboost2(M[sub], genes, tfs, seed=i, out_tsv=adj)
        run_ctx(adj, loom, reg)
        run_csvs.append(reg)
    pd.concat(sub_rows, axis=1).to_csv(TAB / "Fig3_GRN_subsample_cells_per_state.csv")
    gs, reg_tab = consensus_regulons(run_csvs)
    auc_file = TAB / f"Fig3_AUCell_regulons_{SCN_TAG}.csv"
    if auc_file.exists() and "aucell" not in FORCE:
        auc = pd.read_csv(auc_file, index_col=0)
    else:
        auc = aucell(M_all, genes_all, gs)
        auc.index = my.obs_names
        auc.to_csv(auc_file)
    rss = regulon_specificity_scores(auc, my.obs["myeloid_state"])
    rss.to_csv(TAB / "Fig3_RSS.csv")
    return auc, rss, reg_tab


# ==============================================================================
# 5. FIG 3A-D  (regulon heatmap, RSS, regulon UMAPs)
# ==============================================================================
def regulon_selection(rss, states):
    sel, owner = [], {}
    for s in states:
        for r in rss.loc[s].sort_values(ascending=False).index[:N_TOP_RSS_PER_STATE]:
            if r not in owner:
                owner[r] = s
                sel.append(r)
    return sel, owner


def fig3A(my, auc, rss):
    states = [s for s in MYELOID_ORDER if s in rss.index]
    regs, _ = regulon_selection(rss, states)
    z = (auc - auc.mean()) / auc.std(ddof=0)
    med = z.groupby(my.obs["myeloid_state"].values).median().reindex(states)[regs]
    med.to_csv(TAB / "Fig3A_median_regulon_z.csv")
    fig, ax = plt.subplots(figsize=(7.1, 2.1))
    im = ax.imshow(med.values, aspect="auto", cmap="viridis", vmin=-2, vmax=2)
    ax.set_yticks(range(len(states)))
    ax.set_yticklabels(states, fontsize=6)
    ax.set_xticks(range(len(regs)))
    ax.set_xticklabels([tf_of(r) for r in regs], rotation=90, fontsize=5.5)
    ax.set_title("SCENIC GC myeloid cells", fontsize=8)
    cb = plt.colorbar(im, ax=ax, fraction=0.015, pad=0.01, ticks=[-2, 0, 2])
    cb.set_label("Z-score", fontsize=6)
    cb.outline.set_linewidth(0.4)
    panel_label(ax, "A", x=-0.08, y=1.0)
    savefig(fig, "Fig3A_regulon_heatmap")


def fig3_rss(rss, state, letter):
    v = rss.loc[state].sort_values(ascending=False)
    x = np.arange(1, len(v) + 1)
    fig, ax = plt.subplots(figsize=(2.2, 2.5))
    ax.scatter(x, v.values, s=5, color=RSS_BLUE, lw=0, zorder=2, rasterized=True)
    top = v.index[:10]
    ax.scatter(x[:10], v.values[:10], s=9, color=RSS_RED, lw=0, zorder=3)
    texts = [ax.text(x[i], v.values[i], tf_of(r), fontsize=5.5, color=RSS_GREEN, fontweight="bold")
             for i, r in enumerate(top)]
    adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle="-", color=RSS_GREEN, lw=0.4),
                expand=(1.3, 1.5))
    ax.set_xticks([])
    ax.set_xlabel("Regulon")
    ax.set_ylabel("Specificity Score")
    ax.set_title(state.replace("_", " "), fontsize=7)
    for s_ in ax.spines.values():
        s_.set_visible(True)
    panel_label(ax, letter, x=-0.22)
    savefig(fig, f"Fig3{letter}_RSS_{state}")


def fig3D(my, auc, rss, sap_high):
    regs = []
    for s in sap_high:
        for r in rss.loc[s].sort_values(ascending=False).index:
            if r not in regs:
                regs.append(r)
                break
    for s in sap_high:
        for r in rss.loc[s].sort_values(ascending=False).index[:6]:
            if len(regs) >= 6:
                break
            if r not in regs:
                regs.append(r)
    lite = ad.AnnData(obs=pd.DataFrame(index=my.obs_names), obsm={"X_umap": my.obsm["X_umap"]})
    for r in regs:
        v = auc[r].values
        hi = np.percentile(v, 99.5)
        lite.obs[r] = np.clip((v - v.min()) / (hi - v.min() + 1e-12), 0, 1)
    fig, axes = plt.subplots(3, 2, figsize=(4.4, 5.6), gridspec_kw={"wspace": 0.05, "hspace": 0.25})
    for ax, r in zip(axes.ravel(), regs):
        sc.pl.umap(lite, color=r, ax=ax, show=False, frameon=False, size=4, cmap=AUC_CMAP,
                   vmin=0, vmax=1, colorbar_loc=None, sort_order=True, add_outline=True,
                   outline_width=(0.2, 0.05), outline_color=("black", "white"), title="")
        ax.set_title(f"{tf_of(r)} Regulon Activity", fontsize=6.5, fontweight="normal")
        rasterize_points(ax)
    for ax in axes.ravel()[len(regs):]:
        ax.axis("off")
    cax = fig.add_axes([0.35, 0.06, 0.3, 0.012])
    cb = mpl.colorbar.ColorbarBase(cax, cmap=AUC_CMAP, orientation="horizontal", ticks=[0, 1])
    cb.ax.set_xticklabels(["0", "1"], fontsize=6)
    cb.set_label("Normalized\nAUC Score", fontsize=6)
    cb.outline.set_linewidth(0.4)
    panel_label(axes[0, 0], "D", x=0.0)
    savefig(fig, "Fig3D_regulon_activity_umaps")
    return regs


# ==============================================================================
# 6. FIG 3J  regulon activity vs SAP WITHIN states (tumour level)
# ==============================================================================
def regulon_vs_sap(my, auc, rss, sap):
    states = [s for s in TRAJ_STATES if s in rss.index]
    regs, _ = regulon_selection(rss, states)
    o = my.obs[["sample_id", "myeloid_state", "dataset"]].copy()
    o["sample_id"] = o.sample_id.astype(str)
    rows = []
    for s in states:
        m = (o.myeloid_state == s).values
        per = auc.loc[m, regs].groupby(o.sample_id.values[m]).mean()
        n = o[m].groupby("sample_id").size()
        per = per.loc[n[n >= MIN_CELLS_STATE_TUMOUR].index.intersection(sap.index)]
        if len(per) < 10:
            continue
        meta = sap.loc[per.index, ["dataset", "SAP_rin"]]
        rhs = "C(dataset) + SAP_rin" if meta.dataset.nunique() > 1 else "SAP_rin"
        for r in regs:
            y = per[r]
            if y.std() == 0:
                continue
            d = meta.assign(y=((y - y.mean()) / y.std(ddof=1)).values)
            f = smf.ols(f"y ~ {rhs}", data=d).fit()
            rows.append(dict(state=s, regulon=r, beta=f.params["SAP_rin"], p=f.pvalues["SAP_rin"],
                             n_tumours=len(d)))
    res = pd.DataFrame(rows)
    res["fdr"] = multipletests(res.p, method="fdr_bh")[1]
    res.to_csv(TAB / "Fig3J_regulon_vs_SAP_within_state.csv", index=False)
    top = res.groupby("regulon").fdr.min().sort_values().index[:20]
    sub = res[res.regulon.isin(top)]
    piv_b = sub.pivot(index="regulon", columns="state", values="beta").reindex(index=top,
                                                                                   columns=states)
    piv_q = sub.pivot(index="regulon", columns="state", values="fdr").reindex(index=top, columns=states)
    fig, ax = plt.subplots(figsize=(2.6, 0.17 * len(top) + 1.0))
    vmax = np.nanmax(np.abs(piv_b.values))
    for i in range(piv_b.shape[0]):
        for j in range(piv_b.shape[1]):
            b, q = piv_b.values[i, j], piv_q.values[i, j]
            if np.isnan(b):
                continue
            ax.scatter(j, i, s=8 + 22 * min(-np.log10(q), 3), c=[b], cmap="RdBu_r", vmin=-vmax, vmax=vmax,
                       edgecolors="black" if q < 0.05 else "none", linewidths=0.5)
    sm = mpl.cm.ScalarMappable(cmap="RdBu_r", norm=mpl.colors.Normalize(-vmax, vmax))
    cb = plt.colorbar(sm, ax=ax, shrink=0.5, pad=0.02)
    cb.set_label(f"\u03b2 per SD {AXIS_LABEL}", fontsize=6)
    ax.set_xticks(range(len(states)))
    ax.set_xticklabels([s.replace("_", " ") for s in states], rotation=90, fontsize=6)
    ax.set_yticks(range(len(top)))
    ax.set_yticklabels([tf_of(r) for r in top], fontsize=6)
    ax.set_ylim(len(top) - 0.5, -0.5)
    ax.set_xlim(-0.6, len(states) - 0.4)
    ax.set_title(f"Regulon activity vs {AXIS_LABEL}\n(within state, tumour level)", fontsize=7)
    handles = [Line2D([0], [0], marker="o", ls="", color="grey", markersize=np.sqrt(8 + 22 * k),
                      label=f"FDR 1e-{k}" if k else "FDR 1") for k in [0, 1, 2]]
    handles.append(Line2D([0], [0], marker="o", ls="", markerfacecolor="white", markeredgecolor="black",
                          label="FDR < 0.05"))
    ax.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, -0.32), ncol=2, frameon=False,
              fontsize=5.5)
    panel_label(ax, "J", x=-0.35)
    savefig(fig, "Fig3J_regulon_vs_SAP")
    return res


# ==============================================================================
# 7. TRAJECTORY  (Palantir + CellRank)
# ==============================================================================
def extreme_cell(adata, mask, score, rep="X_scVI", q=0.99, rng=None):
    """Medoid (in scVI space) of the cells with the top 1% identity score within `mask`;
    with `rng`, a random cell from that top set (used for the robustness analysis)."""
    idx = np.where(mask)[0]
    s = score[idx]
    top = idx[s >= np.quantile(s, q)]
    if rng is not None:
        return adata.obs_names[rng.choice(top)]
    X = adata.obsm[rep][top]
    return adata.obs_names[top[np.argmin(np.linalg.norm(X - X.mean(0), axis=1))]]


def run_palantir_once(traj, early, terminal_series, use_early=False):
    kw = dict(num_waypoints=N_WAYPOINTS, terminal_states=terminal_series,
              use_early_cell_as_start=use_early)
    try:
        return palantir.core.run_palantir(traj, early, seed=SEED, **kw)
    except TypeError:
        return palantir.core.run_palantir(traj, early, **kw)


def build_trajectory(my):
    out = H5 / "Fig3_trajectory.h5ad"
    if out.exists() and "traj" not in FORCE:
        traj = read_h5ad_compat(out)
        return traj, pd.read_csv(TAB / "Fig3_fate_probabilities.csv", index_col=0)
    traj = my[my.obs.myeloid_state.isin(TRAJ_STATES)].copy()
    # CellRank (GPCCA, lineage drivers) requires a categorical cluster key
    traj.obs["myeloid_state"] = pd.Categorical(traj.obs["myeloid_state"].astype(str), categories=TRAJ_STATES)
    sc.pp.neighbors(traj, use_rep="X_scVI", n_neighbors=30, random_state=SEED)
    sc.tl.umap(traj, min_dist=0.3, random_state=SEED)
    try:
        palantir.utils.run_diffusion_maps(traj, n_components=10, pca_key="X_scVI", seed=SEED)
    except TypeError:
        palantir.utils.run_diffusion_maps(traj, n_components=10, pca_key="X_scVI")
    palantir.utils.determine_multiscale_space(traj)
    # identity scores
    sc.tl.score_genes(traj, [g for g in MONO_GENES if g in traj.var_names], score_name="_mono", random_state=SEED)
    sc.tl.score_genes(traj, [g for g in MACRO_GENES if g in traj.var_names], score_name="_macro", random_state=SEED)
    mono_id = (traj.obs._mono - traj.obs._macro).values
    st = traj.obs.myeloid_state.values
    early = extreme_cell(traj, st == ORIGIN_STATE, mono_id)
    term_cells = {}
    for s in TERMINAL_STATES:
        sc.tl.score_genes(traj, [g for g in STATE_PANELS[s] if g in traj.var_names], score_name="_t",
                          random_state=SEED)
        term_cells[extreme_cell(traj, st == s, traj.obs._t.values)] = s
    terminal_series = pd.Series(term_cells)
    log(f"  Palantir: start {early} ({ORIGIN_STATE}); terminal anchors {terminal_series.to_dict()}")
    run_palantir_once(traj, early, terminal_series)
    # robustness to the start cell. By default Palantir snaps the start cell to the nearest
    # diffusion-space boundary cell, so neighbouring candidates give identical pseudotimes. The test
    # therefore forces the exact start cell (use_early_cell_as_start=True) and uses genuinely
    # different starts: the identity medoid itself and two random classical monocytes.
    rng = np.random.default_rng(SEED)
    mono_idx = np.where(st == ORIGIN_STATE)[0]
    alts = [("medoid top-1% (exact start)", early)] + \
           [(f"random classical monocyte #{k + 1} (exact start)", traj.obs_names[rng.choice(mono_idx)])
            for k in range(2)]
    rob = [dict(start="main (Palantir boundary start)", cell=early, spearman_vs_main=1.0)]
    for label, cell in alts:
        tmp = traj.copy()
        run_palantir_once(tmp, cell, terminal_series, use_early=True)
        rho = stats.spearmanr(tmp.obs.palantir_pseudotime, traj.obs.palantir_pseudotime)[0]
        rob.append(dict(start=label, cell=cell, spearman_vs_main=float(rho)))
        del tmp
    pd.DataFrame(rob).to_csv(TAB / "Fig3_pseudotime_robustness.csv", index=False)
    log(f"  pseudotime robustness (Spearman vs main): {[round(r['spearman_vs_main'], 3) for r in rob]}")
    # CellRank 2
    pk = cr.kernels.PseudotimeKernel(traj, time_key="palantir_pseudotime").compute_transition_matrix()
    ck = cr.kernels.ConnectivityKernel(traj).compute_transition_matrix()
    kernel = PK_WEIGHT * pk + CK_WEIGHT * ck
    g = cr.estimators.GPCCA(kernel)
    pt = traj.obs.palantir_pseudotime
    term = {}
    for s in TERMINAL_STATES:
        m = traj.obs.myeloid_state == s
        thr = pt[m].quantile(1 - TERMINAL_TOP_FRAC)
        term[s] = list(traj.obs_names[m & (pt >= thr)])
    g.set_terminal_states(states=term)
    g.compute_fate_probabilities()
    fp = g.fate_probabilities
    fate = pd.DataFrame(np.asarray(fp), index=traj.obs_names, columns=list(fp.names))
    fate.to_csv(TAB / "Fig3_fate_probabilities.csv")
    for c in fate.columns:
        traj.obs[f"fate_{c}"] = fate[c].values
    # lineage drivers
    drv = []
    for L in fate.columns:
        d = g.compute_lineage_drivers(lineages=[L], cluster_key="myeloid_state", clusters=TRAJ_STATES)
        d = d[[f"{L}_corr", f"{L}_qval"]].rename(columns={f"{L}_corr": "corr", f"{L}_qval": "qval"})
        d["lineage"] = L
        drv.append(d.rename_axis("gene").reset_index())
    pd.concat(drv).to_csv(TAB / "Fig3_lineage_drivers.csv", index=False)
    T = kernel.transition_matrix
    sp.save_npz(H5 / "Fig3_transition_matrix.npz", sp.csr_matrix(T))
    traj.obs.drop(columns=[c for c in ["_mono", "_macro", "_t"] if c in traj.obs], inplace=True)
    safe_write(traj, out)
    return traj, fate


# ==============================================================================
# 8. FIG 3E-I, K
# ==============================================================================
def fig3E(traj):
    lite = ad.AnnData(obs=traj.obs[["palantir_pseudotime"]].copy(), obsm={"X_umap": traj.obsm["X_umap"]})
    fig, ax = plt.subplots(figsize=(2.6, 2.4))
    sc.pl.umap(lite, color="palantir_pseudotime", ax=ax, show=False, frameon=False, size=4,
               cmap="Spectral_r", title="Pseudotime", colorbar_loc="right")
    rasterize_points(ax)
    umap_axis_arrows(ax)
    panel_label(ax, "E", x=0.0)
    savefig(fig, "Fig3E_pseudotime_umap")


def fig3F(traj):
    T = sp.load_npz(H5 / "Fig3_transition_matrix.npz").tocsr()
    U = np.asarray(traj.obsm["X_umap"])
    V = T @ U - U
    from sklearn.neighbors import NearestNeighbors
    ng = 45
    xs = np.linspace(U[:, 0].min(), U[:, 0].max(), ng)
    ys = np.linspace(U[:, 1].min(), U[:, 1].max(), ng)
    gx, gy = np.meshgrid(xs, ys)
    grid = np.c_[gx.ravel(), gy.ravel()]
    nn = NearestNeighbors(n_neighbors=30).fit(U)
    dist, ind = nn.kneighbors(grid)
    scale = np.mean([xs[1] - xs[0], ys[1] - ys[0]]) * 1.5
    w = np.exp(-(dist / scale) ** 2)
    wsum = w.sum(1)
    gv = (V[ind] * w[..., None]).sum(1) / np.maximum(wsum, 1e-12)[:, None]
    gv[wsum < np.percentile(wsum, 20)] = np.nan               # no arrows where there are no cells
    su, sv = gv[:, 0].reshape(ng, ng), gv[:, 1].reshape(ng, ng)
    lite = ad.AnnData(obs=traj.obs[["myeloid_state"]].copy(), obsm={"X_umap": U})
    set_palette(lite, "myeloid_state")
    fig, ax = plt.subplots(figsize=(2.8, 2.6))
    sc.pl.umap(lite, color="myeloid_state", ax=ax, show=False, frameon=False, size=4, alpha=0.6,
               legend_loc="on data", legend_fontsize=5.5, legend_fontweight="bold", legend_fontoutline=1.5,
               title="")
    rasterize_points(ax)
    ax.streamplot(xs, ys, su, sv, color="black", linewidth=0.5, density=1.3, arrowsize=0.6,
                  arrowstyle="-|>", zorder=5)
    umap_axis_arrows(ax)
    panel_label(ax, "F", x=0.0)
    savefig(fig, "Fig3F_cellrank_streamlines")


def binned(values, pt, weights, bins):
    idx = np.clip(np.digitize(pt, bins) - 1, 0, len(bins) - 2)
    nb = len(bins) - 1
    wsum = np.bincount(idx, weights=weights, minlength=nb)
    if values.ndim == 1:
        return np.bincount(idx, weights=values * weights, minlength=nb) / np.maximum(wsum, 1e-12)
    return np.vstack([np.bincount(idx, weights=values[:, j] * weights, minlength=nb) / np.maximum(wsum, 1e-12)
                      for j in range(values.shape[1])])


def fig3_drivers(traj, fate, lineage, letter, tf_names):
    drv = pd.read_csv(TAB / "Fig3_lineage_drivers.csv")
    expr_ok = np.asarray((traj.X > 0).mean(axis=0)).ravel() >= 0.05
    ok = set(traj.var_names[expr_ok])
    # bracket access: `drv.corr` would be the DataFrame.corr() METHOD, not the column
    d = drv[(drv["lineage"] == lineage) & (drv["qval"] < 0.05) & (drv["corr"] > 0) & drv["gene"].isin(ok)]
    genes = d.sort_values("corr", ascending=False).gene.head(N_DRIVERS).tolist()
    if not genes:
        log(f"  no significant drivers for {lineage}")
        return
    w = fate[lineage].values
    sel = w >= np.quantile(w, 0.5)                         # cells biased toward this fate
    pt = traj.obs.palantir_pseudotime.values
    bins = np.linspace(pt[sel].min(), pt[sel].max(), 101)
    X = traj[:, genes].X
    X = X.toarray() if sp.issparse(X) else np.asarray(X)
    M = binned(X[sel], pt[sel], np.ones(sel.sum()), bins)
    M = gaussian_filter1d(M, sigma=3, axis=1, mode="nearest")
    M = (M - M.min(1, keepdims=True)) / (M.max(1, keepdims=True) - M.min(1, keepdims=True) + 1e-12)
    order = np.argsort(np.argmax(M, axis=1))
    M, genes = M[order], [genes[i] for i in order]
    fb = gaussian_filter1d(binned(w[sel], pt[sel], np.ones(sel.sum()), bins), 2, mode="nearest")
    idx = np.clip(np.digitize(pt[sel], bins) - 1, 0, 99)
    stt = traj.obs.myeloid_state.values[sel]
    dom = [pd.Series(stt[idx == b]).mode().iloc[0] if (idx == b).any() else None for b in range(100)]
    fig = plt.figure(figsize=(2.9, 0.13 * len(genes) + 0.9))
    gs = gridspec.GridSpec(3, 2, height_ratios=[0.06, 0.06, 1], width_ratios=[1, 0.04], hspace=0.05, wspace=0.05)
    a0, a1, a2 = fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[1, 0]), fig.add_subplot(gs[2, 0])
    a0.imshow(np.array([[mpl.colors.to_rgb(MYELOID_PALETTE.get(s, "#FFFFFF")) if s else (1, 1, 1)
                         for s in dom]]), aspect="auto")
    im1 = a1.imshow(fb[None, :], aspect="auto", cmap="Greys", vmin=0, vmax=1)
    im2 = a2.imshow(M, aspect="auto", cmap="viridis", vmin=0, vmax=1)
    for a in (a0, a1):
        a.set_xticks([]); a.set_yticks([])
    a0.set_title(lineage.replace("_", " "), fontsize=7)
    a1.set_ylabel("Fate", rotation=0, fontsize=5.5, ha="right", va="center")
    a2.set_yticks(range(len(genes)))
    a2.set_yticklabels([("* " if g in tf_names else "") + g for g in genes], fontsize=5.5, fontstyle="italic")
    for t, g in zip(a2.get_yticklabels(), genes):
        if g in tf_names:
            t.set_color(RSS_RED)
    xt = np.linspace(0, 99, 5)
    a2.set_xticks(xt)
    a2.set_xticklabels([f"{v:.2f}" for v in np.linspace(bins[0], bins[-1], 5)], fontsize=5.5)
    a2.set_xlabel("palantir_pseudotime", fontsize=6)
    cax = fig.add_subplot(gs[2, 1])
    cb = plt.colorbar(im2, cax=cax, ticks=[0, 1])
    cb.set_label("scaled expression", fontsize=5.5)
    cb.outline.set_linewidth(0.4)
    fig.add_subplot(gs[0:2, 1]).axis("off")
    panel_label(a0, letter, x=-0.06, y=0.9)
    savefig(fig, f"Fig3{letter}_drivers_{lineage}")
    return genes


def fig3I(traj, fate, genes):
    genes = [g for g in dict.fromkeys(genes) if g in traj.var_names][:6]
    pt = traj.obs.palantir_pseudotime.values
    bins = np.linspace(pt.min(), pt.max(), 41)
    mids = (bins[:-1] + bins[1:]) / 2
    X = traj[:, genes].X
    X = X.toarray() if sp.issparse(X) else np.asarray(X)
    rng = np.random.default_rng(SEED)
    fig, axes = plt.subplots(2, 3, figsize=(5.4, 3.2), gridspec_kw={"wspace": 0.35, "hspace": 0.55})
    for ax, (j, g) in zip(axes.ravel(), enumerate(genes)):
        for L in fate.columns:
            w = fate[L].values
            mu = gaussian_filter1d(binned(X[:, j], pt, w, bins), 1.5, mode="nearest")
            boots = []
            for _ in range(50):
                b = rng.integers(0, len(pt), len(pt))
                boots.append(gaussian_filter1d(binned(X[b, j], pt[b], w[b], bins), 1.5, mode="nearest"))
            lo, hi = np.percentile(boots, [2.5, 97.5], axis=0)
            c = MYELOID_PALETTE.get(L, "grey")
            ax.fill_between(mids, lo, hi, color=c, alpha=0.2, lw=0)
            ax.plot(mids, mu, color=c, lw=1.0, label=L.replace("_", " "))
        ax.set_title(g, fontsize=7, fontstyle="italic")
        ax.set_xlabel("palantir_pseudotime", fontsize=5.5)
        ax.set_ylabel("expression", fontsize=5.5)
        for s_ in ["top", "right"]:
            ax.spines[s_].set_visible(False)
    for ax in axes.ravel()[len(genes):]:
        ax.axis("off")
    axes[0, 0].legend(frameon=False, fontsize=5, loc="upper left", title="Fate", title_fontsize=5.5)
    panel_label(axes[0, 0], "I", x=-0.3)
    savefig(fig, "Fig3I_gene_trends")


def fig3K(traj, fate, sap):
    pool = traj.obs.myeloid_state.isin([ORIGIN_STATE, "TAM_Transitional"]).values
    o = traj.obs.loc[pool, ["sample_id"]].astype(str)
    f = fate.loc[pool].groupby(o.sample_id.values).mean()
    n = o.groupby("sample_id").size()
    f = f.loc[n[n >= MIN_CELLS_STATE_TUMOUR].index.intersection(sap.index)]
    meta = sap.loc[f.index, ["dataset", "SAP_rin"]]
    rows = []
    for L in fate.columns:
        y = f[L]
        d = meta.assign(y=((y - y.mean()) / y.std(ddof=1)).values)
        fit = smf.ols("y ~ C(dataset) + SAP_rin", data=d).fit()
        rows.append(dict(lineage=L, beta=fit.params["SAP_rin"], p=fit.pvalues["SAP_rin"], n_tumours=len(d),
                         spearman=stats.spearmanr(meta.SAP_rin, y)[0]))
    res = pd.DataFrame(rows)
    res["fdr"] = multipletests(res.p, method="fdr_bh")[1]
    res.to_csv(TAB / "Fig3K_fate_bias_vs_SAP.csv", index=False)
    fig, axes = plt.subplots(1, len(fate.columns), figsize=(1.75 * len(fate.columns), 1.9),
                             gridspec_kw={"wspace": 0.5})
    for ax, L in zip(np.atleast_1d(axes), fate.columns):
        for dsn, c in DATASET_PALETTE.items():
            m = meta.dataset == dsn
            ax.scatter(meta.SAP_rin[m], f.loc[m.index[m], L], s=8, color=c, edgecolor="black", lw=0.2, zorder=3)
        b = np.polyfit(meta.SAP_rin, f[L], 1)
        xx = np.linspace(meta.SAP_rin.min(), meta.SAP_rin.max(), 50)
        ax.plot(xx, np.polyval(b, xx), color="black", lw=0.7)
        r = res.set_index("lineage").loc[L]
        ax.set_title(L.replace("_", " "), fontsize=6.5, color=MYELOID_PALETTE.get(L, "black"), pad=11)
        ax.text(0.5, 1.02, f"{fdr_stars(r.fdr)}  FDR = {r.fdr:.2g}", transform=ax.transAxes, ha="center",
                va="bottom", fontsize=5.5, fontweight="bold" if r.fdr < 0.05 else "normal")
        ax.set_xlabel(f"{AXIS_LABEL} (rank-normal)", fontsize=6)
        ax.set_ylabel("Mean fate probability", fontsize=6)
        for s_ in ["top", "right"]:
            ax.spines[s_].set_visible(False)
    handles = [Line2D([0], [0], marker="o", ls="", markerfacecolor=c, markeredgecolor="black",
                      markeredgewidth=0.2, markersize=4, label=d) for d, c in DATASET_PALETTE.items()]
    fig.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, 1.02), ncol=2, frameon=False, fontsize=6)
    panel_label(np.atleast_1d(axes)[0], "K", x=-0.3, y=1.15)
    savefig(fig, "Fig3K_fate_bias_vs_SAP")
    log(f"  fate bias vs SAP: {res.round(4).to_dict('records')}")
    return res


def figS6_robustness():
    rob = pd.read_csv(TAB / "Fig3_pseudotime_robustness.csv")
    reg = pd.read_csv(TAB / f"Fig3_consensus_regulons_{SCN_TAG}.csv")
    fig, axes = plt.subplots(1, 2, figsize=(4.6, 1.9), gridspec_kw={"wspace": 0.5})
    axes[0].bar(range(len(rob)), rob.spearman_vs_main, color="#7F7F7F")
    axes[0].set_xticks(range(len(rob)))
    axes[0].set_xticklabels(["main", "medoid\n(exact)", "random 1", "random 2"][:len(rob)], fontsize=5.5)
    axes[0].set_ylim(0, 1.05)
    axes[0].set_ylabel("Spearman \u03c1 vs main pseudotime")
    axes[0].set_title("Start-cell robustness", fontsize=7)
    axes[1].hist(reg.jaccard_mean.dropna(), bins=20, color="#4C6FB5")
    axes[1].set_xlabel("Mean Jaccard of target sets\nacross GRNBoost2 runs")
    axes[1].set_ylabel("Regulons")
    axes[1].set_title(f"{len(reg)} consensus regulons", fontsize=7)
    for ax in axes:
        for s_ in ["top", "right"]:
            ax.spines[s_].set_visible(False)
    savefig(fig, "FigS6_scenic_trajectory_robustness", FIG_SUPP)


# ==============================================================================
# 9. MAIN
# ==============================================================================
def main():
    np.random.seed(SEED)
    log("=" * 70)
    my, sap, sap_high, sap_low = load_inputs()

    auc, rss, reg_tab = run_scenic(my)
    tf_names = set(reg_tab.tf)
    fig3A(my, auc, rss)
    rss_states = [s for s in sap_high if s in rss.index][:2]
    for s, letter in zip(rss_states, ["B", "C"]):
        fig3_rss(rss, s, letter)
    d_regs = fig3D(my, auc, rss, rss_states)
    regulon_vs_sap(my, auc, rss, sap)

    traj, fate = build_trajectory(my)
    fig3E(traj)
    fig3F(traj)
    hi_term = [s for s in sap_high if s in TERMINAL_STATES] or ["TAM_Inflammatory"]
    lo_term = [s for s in sap_low if s in TERMINAL_STATES] or ["TAM_C1QC"]
    drv_hi = fig3_drivers(traj, fate, hi_term[0], "G", tf_names) or []
    fig3_drivers(traj, fate, lo_term[0], "H", tf_names)
    data_tfs = [g for g in drv_hi if g in tf_names][:2]
    fig3I(traj, fate, TREND_GENES_FIXED[:3] + data_tfs + TREND_GENES_FIXED[3:])
    fig3K(traj, fate, sap)
    figS6_robustness()

    my.obs = my.obs.join(auc.add_prefix("REG_"))
    safe_write(my, H5 / "Fig3_myeloid_with_regulons.h5ad")
    summary = {
        "n_cells": int(my.n_obs), "n_consensus_regulons": int(len(reg_tab)),
        "grn_runs": N_GRN_RUNS, "consensus_min_runs": CONSENSUS_MIN_RUNS,
        "rss_states": rss_states, "umap_regulons": d_regs,
        "trajectory_states": TRAJ_STATES, "terminal_states": TERMINAL_STATES,
        "pseudotime_robustness": pd.read_csv(TAB / "Fig3_pseudotime_robustness.csv").spearman_vs_main.round(3).tolist(),
        "software": {"pyscenic": __import__("pyscenic").__version__, "palantir": palantir.__version__,
                     "cellrank": cr.__version__, "scanpy": sc.__version__},
    }
    json.dump(summary, open(TAB / "Fig3_methods_numbers.json", "w"), indent=2)
    log(f"DONE. {summary}")


if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python
# coding: utf-8
"""
GC_Fig3I_redraw.py -- layout-only redraw of Fig3I (gene trends along pseudotime).
2 x 2 panels; the fate legend sits BELOW the panels (no overlap with the curves).
Nothing is recomputed upstream: pseudotime and fate probabilities are read from the saved
trajectory; the smoothing and the seeded bootstrap are identical to GC_TAM_Figure3.py, so the
curves and confidence bands are exactly the same. Overwrites figures/main/Fig3I_gene_trends.*
"""
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
import scanpy as sc
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.ndimage import gaussian_filter1d

import os
OUT = Path(os.environ.get("GC_BASE", "/home/arshia/payanname")) / "GC_Fig3_outputs"
FIG = OUT / "figures" / "main"
SEED = 0
GENES = ["CCR2", "SLAMF9", "IL1B", "C1QC"]          # the four genes of the published panel, same order
PALETTE = {"TAM_C1QC": "#6A3D9A", "TAM_Inflammatory": "#FF9E4A", "TAM_SPP1": "#E31A1C"}


def _font():
    from matplotlib import font_manager
    avail = {f.name for f in font_manager.fontManager.ttflist}
    return next((f for f in ["Arial", "Helvetica", "Liberation Sans", "Nimbus Sans"] if f in avail), "DejaVu Sans")


mpl.rcParams.update({
    "font.family": "sans-serif", "font.sans-serif": [_font()], "font.size": 7,
    "axes.titlesize": 8, "axes.titleweight": "bold", "axes.labelsize": 7,
    "xtick.labelsize": 6, "ytick.labelsize": 6, "legend.fontsize": 6,
    "axes.linewidth": 0.6, "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "xtick.major.size": 2.5, "ytick.major.size": 2.5,
    "pdf.fonttype": 42, "ps.fonttype": 42, "svg.fonttype": "none",
    "savefig.dpi": 600, "savefig.bbox": "tight", "savefig.pad_inches": 0.02,
})


def binned(values, pt, weights, bins):
    idx = np.clip(np.digitize(pt, bins) - 1, 0, len(bins) - 2)
    nb = len(bins) - 1
    wsum = np.bincount(idx, weights=weights, minlength=nb)
    return np.bincount(idx, weights=values * weights, minlength=nb) / np.maximum(wsum, 1e-12)


def main():
    traj = sc.read_h5ad(OUT / "h5ad" / "Fig3_trajectory.h5ad")
    fate = pd.read_csv(OUT / "tables" / "Fig3_fate_probabilities.csv", index_col=0).loc[traj.obs_names]
    genes = [g for g in GENES if g in traj.var_names]
    pt = traj.obs["palantir_pseudotime"].values
    bins = np.linspace(pt.min(), pt.max(), 41)
    mids = (bins[:-1] + bins[1:]) / 2
    X = traj[:, genes].X
    X = X.toarray() if sp.issparse(X) else np.asarray(X)
    rng = np.random.default_rng(SEED)                 # same seed and draw order as the main script

    fig, axes = plt.subplots(2, 2, figsize=(3.8, 3.4), gridspec_kw={"wspace": 0.45, "hspace": 0.65})
    for ax, (j, g) in zip(axes.ravel(), enumerate(genes)):
        for L in fate.columns:
            w = fate[L].values
            mu = gaussian_filter1d(binned(X[:, j], pt, w, bins), 1.5, mode="nearest")
            boots = []
            for _ in range(50):
                b = rng.integers(0, len(pt), len(pt))
                boots.append(gaussian_filter1d(binned(X[b, j], pt[b], w[b], bins), 1.5, mode="nearest"))
            lo, hi = np.percentile(boots, [2.5, 97.5], axis=0)
            c = PALETTE.get(L, "grey")
            ax.fill_between(mids, lo, hi, color=c, alpha=0.2, lw=0)
            ax.plot(mids, mu, color=c, lw=1.0)
        ax.set_title(g, fontsize=7, fontstyle="italic")
        ax.set_xlabel("palantir_pseudotime", fontsize=5.5)
        ax.set_ylabel("expression", fontsize=5.5)
        for s_ in ["top", "right"]:
            ax.spines[s_].set_visible(False)
    for ax in axes.ravel()[len(genes):]:
        ax.axis("off")

    handles = [Line2D([0], [0], color=PALETTE.get(L, "grey"), lw=1.2, label=L.replace("_", " "))
               for L in fate.columns]
    fig.legend(handles=handles, title="Fate", loc="upper center", bbox_to_anchor=(0.5, 0.0),
               ncol=len(handles), frameon=False, fontsize=5.5, title_fontsize=6, handlelength=1.6)
    axes[0, 0].text(-0.3, 1.02, "I", fontsize=10, fontweight="bold", va="bottom", ha="right",
                    transform=axes[0, 0].transAxes)
    for ext in ["pdf", "svg", "png"]:
        fig.savefig(FIG / f"Fig3I_gene_trends.{ext}")
    plt.close(fig)
    print("replaced figures/main/Fig3I_gene_trends (pdf/svg/png)")


if __name__ == "__main__":
    main()